In [9]:
import os
import json
import math
import random
import string
import numpy as np
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download, login

# Configurations and hyperparameters

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = ""
if HF_TOKEN:
    login(token=HF_TOKEN)

# Paths for Phase 1 Checkpoint & Tokenizer
TOKENIZER_PATH = "/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json"
CHECKPOINT_PATH = "/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt"

# Phase 2 Hyperparameters
MAX_SEQ_LEN = 512
STRIDE = 128
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4      # adapter (LoRMA) params
HEAD_LR = 1e-3            # classifier head
EPOCHS = 10
LORA_RANK = 32
LORA_ALPHA = 64.0
USE_VAL = True            # True: train fold1, validate on fold2.  False: final run on folds 1+2

# 12 Official Labels
LABEL_NAMES = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
    "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
# BIO Scheme (25 tags total)
ID2LABEL = {0: "O"}
LABEL2ID = {"O": 0}
for idx, lbl in enumerate(LABEL_NAMES):
    ID2LABEL[1 + 2 * idx] = f"B-{lbl}"
    LABEL2ID[f"B-{lbl}"] = 1 + 2 * idx
    ID2LABEL[2 + 2 * idx] = f"I-{lbl}"
    LABEL2ID[f"I-{lbl}"] = 2 + 2 * idx
NUM_CLASSES = len(ID2LABEL)
IGNORE_INDEX = -100
print(f"Device: {DEVICE} | Classes: {NUM_CLASSES} | USE_VAL={USE_VAL}")

Device: cuda | Classes: 25 | USE_VAL=True


In [10]:
SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
END_OF_WORD = "</w>"

class ByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []
        self.merge_rank = {}
        self.itos = []
        self.stoi = {}

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=data["vocab_size"])
        tok.itos = data["itos"]
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.merges = [tuple(p) for p in data["merges"]]
        tok.merge_rank = {p: i for i, p in enumerate(tok.merges)}
        return tok

    def _word_to_symbols(self, word):
        return list(word.encode("utf-8").decode("latin-1")) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        symbols = self._word_to_symbols(word)
        while len(symbols) > 1:
            pairs_here = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            candidates = [(self.merge_rank[p], idx) for idx, p in enumerate(pairs_here) if p in self.merge_rank]
            if not candidates:
                break
            _, idx = min(candidates)
            merged = symbols[idx] + symbols[idx + 1]
            symbols = symbols[:idx] + [merged] + symbols[idx + 2:]
        return symbols

    def encode_with_exact_offsets(self, text: str) -> Tuple[List[int], List[Tuple[int, int]]]:
        """
        Calculates EXACT character-level [start, end) offsets for every single subword token.
        """
        tokens = []
        offsets = []
        i = 0
        n = len(text)

        while i < n:
            while i < n and text[i].isspace():
                i += 1
            if i >= n:
                break
            word_start = i
            while i < n and not text[i].isspace():
                i += 1
            word_end = i
            word = text[word_start:word_end]

            subwords = self._apply_merges_to_word(word)
            curr_pos = word_start

            for sym in subwords:
                clean_sym = sym.replace(END_OF_WORD, "")
                try:
                    sym_chars = clean_sym.encode("latin-1").decode("utf-8", errors="replace")
                except Exception:
                    sym_chars = clean_sym

                sub_len = len(sym_chars)
                sub_start = curr_pos
                sub_end = min(word_end, curr_pos + sub_len)
                curr_pos = sub_end

                t_id = self.stoi.get(sym, self.stoi.get("<unk>", 1))
                tokens.append(t_id)
                offsets.append((sub_start, sub_end))

        return tokens, offsets

# Load tokenizer
tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
print(f"Loaded tokenizer | Vocab size: {tokenizer.vocab_size}")

Loaded tokenizer | Vocab size: 8000


In [11]:
import zipfile
import os

source_dir = '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt'
target_pt = '/kaggle/working/decoder_lm.pt'

# Re-package the folder into the single PyTorch archive file
with zipfile.ZipFile(target_pt, 'w', compression=zipfile.ZIP_STORED) as zf:
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            full_path = os.path.join(root, file)
            # relative path inside the archive
            rel_path = os.path.relpath(full_path, source_dir)
            zf.write(full_path, arcname=rel_path)

print(f"✓ Created valid checkpoint file: {target_pt} ({os.path.getsize(target_pt)/1e6:.1f} MB)")

# Set CHECKPOINT_PATH to this working file
CHECKPOINT_PATH = target_pt

✓ Created valid checkpoint file: /kaggle/working/decoder_lm.pt (421.1 MB)


In [12]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def get_alibi_slopes(n_heads: int):
    def get_slopes_power_of_2(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]

    if math.log2(n_heads).is_integer():
        return torch.tensor(get_slopes_power_of_2(n_heads))
    else:
        closest_pow2 = 2 ** math.floor(math.log2(n_heads))
        slopes_a = get_slopes_power_of_2(closest_pow2)
        slopes_b = get_slopes_power_of_2(2 * closest_pow2)[0::2][:n_heads - closest_pow2]
        return torch.tensor(slopes_a + slopes_b)

class LORMALinear(nn.Module):
    def __init__(self, original_linear: nn.Linear, rank: int = 16, alpha: float = 32.0):
        super().__init__()
        self.original_linear = original_linear
        self.rank = rank
        self.scaling = alpha / rank
        self.original_linear.weight.requires_grad = False

        in_f = original_linear.in_features
        out_f = original_linear.out_features
        self.lorma_down = nn.Linear(in_f, rank, bias=False)
        self.lorma_up = nn.Linear(rank, out_f, bias=False)

        nn.init.kaiming_uniform_(self.lorma_down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_up.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.original_linear(x) + self.lorma_up(self.lorma_down(x)) * self.scaling

class ALiBiBidirectionalAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

        slopes = get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2)
        self.register_buffer("slopes", slopes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)

        # Symmetric bidirectional distance
        pos = torch.arange(S, device=x.device)
        dist = -torch.abs(pos.unsqueeze(0) - pos.unsqueeze(1)).float()
        alibi_bias = self.slopes * dist.unsqueeze(0)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=alibi_bias,
            dropout_p=self.dropout if self.training else 0.0
        )
        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        return self.out_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = ALiBiBidirectionalAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class Arch4LNERModel(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 640, n_layers: int = 16,
                 n_heads: int = 8, d_ff: int = 2560, num_classes: int = NUM_CLASSES,
                 dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.drop(self.tok_emb(input_ids))
        for block in self.blocks:
            x = block(x)
        x = self.norm_f(x)
        return self.classifier(x)

    def apply_lorma(self, rank: int = 16, alpha: float = 32.0, device=DEVICE):
        for param in self.parameters():
            param.requires_grad = False

        for block in self.blocks:
            block.attn.q_proj = LORMALinear(block.attn.q_proj, rank=rank, alpha=alpha).to(device)
            block.attn.v_proj = LORMALinear(block.attn.v_proj, rank=rank, alpha=alpha).to(device)
            block.ffn.gate_proj = LORMALinear(block.ffn.gate_proj, rank=rank, alpha=alpha).to(device)
            block.ffn.up_proj = LORMALinear(block.ffn.up_proj, rank=rank, alpha=alpha).to(device)

        for param in self.classifier.parameters():
            param.requires_grad = True

# ---- NEW: adapters on every linear layer ----
def apply_lorma_all(model, rank, alpha, device):
    for p in model.parameters():
        p.requires_grad = False
    for b in model.blocks:
        for n in ["q_proj", "k_proj", "v_proj", "out_proj"]:
            setattr(b.attn, n, LORMALinear(getattr(b.attn, n), rank, alpha).to(device))
        for n in ["gate_proj", "up_proj", "down_proj"]:
            setattr(b.ffn, n, LORMALinear(getattr(b.ffn, n), rank, alpha).to(device))
    for p in model.classifier.parameters():
        p.requires_grad = True
    for p in model.norm_f.parameters():
        p.requires_grad = True

# Instantiate & Load Checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
d_model_ckpt = state_dict["tok_emb.weight"].shape[1]
vocab_size_ckpt = state_dict["tok_emb.weight"].shape[0]
block_indices = [int(k.split(".")[1]) for k in state_dict.keys() if k.startswith("blocks.") and k.split(".")[1].isdigit()]
n_layers_ckpt = max(block_indices) + 1 if block_indices else 16
n_heads = 8 if d_model_ckpt == 640 else 12

model = Arch4LNERModel(vocab_size=vocab_size_ckpt, d_model=d_model_ckpt, n_layers=n_layers_ckpt, n_heads=n_heads).to(DEVICE)
backbone_dict = {k: v for k, v in state_dict.items() if not k.startswith("lm_head") and not k.startswith("head")}
missing, unexpected = model.load_state_dict(backbone_dict, strict=False)
print("MISSING   :", len(missing), missing[:8])
print("UNEXPECTED:", len(unexpected), unexpected[:8])
assert all(m.startswith("classifier.") or m.endswith(".slopes") for m in missing), \
    "Backbone weights did NOT load - fix key names before continuing!"

apply_lorma_all(model, LORA_RANK, LORA_ALPHA, DEVICE)
model.to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Params: {total_params:,} | Trainable Params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

MISSING   : 6 ['classifier.0.weight', 'classifier.0.bias', 'classifier.1.weight', 'classifier.1.bias', 'classifier.4.weight', 'classifier.4.bias']
UNEXPECTED: 0 []
Total Params: 112,059,673 | Trainable Params: 6,805,273 (6.07%)


In [6]:
def download_phase2_file(filename: str) -> str:
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

print("Downloading dataset files...")
f1_text_path = download_phase2_file("Phase_2_LNER/Train/fold1-text.json")
f1_gold_path = download_phase2_file("Phase_2_LNER/Train/fold1-gold.json")
f2_text_path = download_phase2_file("Phase_2_LNER/Train/fold2-text.json")
f2_gold_path = download_phase2_file("Phase_2_LNER/Train/fold2-gold.json")
f3_text_path = download_phase2_file("Phase_2_LNER/Test/fold3-text.json")

with open(f1_text_path, "r", encoding="utf-8") as f: train_texts = json.load(f)
with open(f1_gold_path, "r", encoding="utf-8") as f: train_golds = json.load(f)
with open(f2_text_path, "r", encoding="utf-8") as f: val_texts = json.load(f)
with open(f2_gold_path, "r", encoding="utf-8") as f: val_golds = json.load(f)
with open(f3_text_path, "r", encoding="utf-8") as f: test_texts = json.load(f)

# ---- NEW: held-out validation switch ----
if USE_VAL:
    tr_texts, tr_golds = train_texts, train_golds
else:
    tr_texts = {**train_texts, **val_texts}
    tr_golds = {**train_golds, **val_golds}
print(f"Training docs: {len(tr_texts)} | USE_VAL={USE_VAL}")

# ---- NEW: windows where the last one is always full-length (no tiny padded tail) ----
def window_starts(n, max_len=MAX_SEQ_LEN, stride=STRIDE):
    if n <= max_len:
        return [0]
    starts = list(range(0, n - max_len + 1, max_len - stride))
    if starts[-1] + max_len < n:
        starts.append(n - max_len)
    return starts

class ExactLNERDataset(Dataset):
    def __init__(self, docs: Dict[str, str], golds: Dict[str, List[Any]], tokenizer: ByteBPETokenizer,
                 max_len: int = MAX_SEQ_LEN, stride: int = STRIDE):
        self.samples = []
        for doc_id, text in docs.items():
            token_ids, offsets = tokenizer.encode_with_exact_offsets(text)

            token_labels = [0] * len(token_ids)
            if doc_id in golds:
                for annot in golds[doc_id]:
                    g_start, g_end, g_lbl = annot[0], annot[1], annot[2]
                    first_matched = False
                    for idx, (t_s, t_e) in enumerate(offsets):
                        if max(t_s, g_start) < min(t_e, g_end):
                            if not first_matched:
                                token_labels[idx] = LABEL2ID.get(f"B-{g_lbl}", 0)
                                first_matched = True
                            else:
                                token_labels[idx] = LABEL2ID.get(f"I-{g_lbl}", 0)

            for start_idx in window_starts(len(token_ids), max_len, stride):
                end_idx = min(len(token_ids), start_idx + max_len)
                c_ids = token_ids[start_idx:end_idx]
                c_lbls = token_labels[start_idx:end_idx]

                pad_len = max_len - len(c_ids)
                padded_ids = c_ids + [0] * pad_len
                padded_lbls = c_lbls + [IGNORE_INDEX] * pad_len

                self.samples.append({
                    "input_ids": torch.tensor(padded_ids, dtype=torch.long),
                    "labels": torch.tensor(padded_lbls, dtype=torch.long)
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

train_ds = ExactLNERDataset(tr_texts, tr_golds, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print(f"Total training sliding windows: {len(train_ds)}")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Training docs: 35 | USE_VAL=True
Total training sliding windows: 652


In [13]:
# ---- boundary cleanup: strip punctuation (incl. curly quotes/dashes) at span edges ----
EXTRA = "\u2018\u2019\u201c\u201d\u2013\u2014\u2022\u00b7\u2026"
STRIP_LEAD = " \t\n\r" + string.punctuation + EXTRA
STRIP_TRAIL = " \t\n\r" + string.punctuation + EXTRA

def clean_span(text: str, start: int, end: int, label: str):
    while start < end and text[start] in STRIP_LEAD:
        start += 1
    while end > start and text[end - 1] in STRIP_TRAIL:
        end -= 1
    return start, end, label

def decode_spans_exact(doc_text: str, pred_ids: List[int], offsets: List[Tuple[int, int]]) -> List[List[Any]]:
    spans, cur_l, cs, ce = [], None, -1, -1

    def flush():
        if cur_l is not None:
            s, e, l = clean_span(doc_text, cs, ce, cur_l)
            if e > s:
                spans.append([s, e, l])

    for t_id, (s_c, e_c) in zip(pred_ids, offsets):
        tag = ID2LABEL.get(int(t_id), "O")
        if tag.startswith("B-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        elif tag.startswith("I-") and cur_l == tag[2:]:
            ce = e_c
        elif tag.startswith("I-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        else:
            flush(); cur_l = None
    flush()

    seen, out = set(), []
    for s in spans:
        if tuple(s) not in seen:
            seen.add(tuple(s)); out.append(s)
    return out

# ---- BIO-constrained Viterbi (with optional continuation bonus) ----
NEG = -1e9
ALLOWED = np.ones((NUM_CLASSES, NUM_CLASSES), dtype=bool)     # [prev, cur]
for cur in range(NUM_CLASSES):
    if ID2LABEL[cur].startswith("I-"):
        for prev in range(NUM_CLASSES):
            ALLOWED[prev, cur] = (ID2LABEL[prev] != "O") and (ID2LABEL[prev][2:] == ID2LABEL[cur][2:])
START_OK = np.array([not ID2LABEL[c].startswith("I-") for c in range(NUM_CLASSES)])

def make_trans(cont_bonus=0.0):
    t = np.where(ALLOWED, 0.0, NEG).astype(np.float64)
    for cur in range(NUM_CLASSES):
        if ID2LABEL[cur].startswith("I-"):
            for prev in range(NUM_CLASSES):
                if ALLOWED[prev, cur]:
                    t[prev, cur] += cont_bonus
    return t

def viterbi(logp, trans):
    T, C = logp.shape
    score = logp[0].copy(); score[~START_OK] = NEG
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]

def merge_spans(text, spans, max_gap=0):
    out = []
    for s, e, l in sorted(spans):
        if out and out[-1][2] == l and 0 <= s - out[-1][1] <= max_gap and text[out[-1][1]:s].strip() == "":
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e, l])
    return out

# ---- per-document log-probs, averaged over overlapping windows; padded exactly like training ----
@torch.no_grad()
def doc_logp(text):
    model.eval()
    ids, offs = tokenizer.encode_with_exact_offsets(text)
    if not ids:
        return None, None
    acc = torch.zeros(len(ids), NUM_CLASSES); cnt = torch.zeros(len(ids), 1)
    for s in window_starts(len(ids)):
        e = min(len(ids), s + MAX_SEQ_LEN)
        chunk = ids[s:e]
        x = torch.tensor([chunk + [0] * (MAX_SEQ_LEN - len(chunk))], dtype=torch.long, device=DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            lp = F.log_softmax(model(x)[0].float(), dim=-1)[: len(chunk)].cpu()
        acc[s:e] += lp; cnt[s:e] += 1
    return (acc / cnt).numpy(), offs

def decode_cached(cache, texts, o_bias=0.0, cont_bonus=0.0, max_gap=0):
    trans, out = make_trans(cont_bonus), {}
    for d, (lp, offs) in cache.items():
        if lp is None:
            out[d] = []
            continue
        lp = lp.astype(np.float64).copy()
        lp[:, 0] += o_bias
        sp = decode_spans_exact(texts[d], viterbi(lp, trans), offs)
        out[d] = merge_spans(texts[d], sp, max_gap) if max_gap > 0 else sp
    return out

def strict_macro_f1(pred, gold, labels=LABEL_NAMES):
    f1s = []
    for lbl in labels:
        tp = fp = fn = 0
        for d in gold:
            g = {(a[0], a[1]) for a in gold[d] if a[2] == lbl}
            p = {(a[0], a[1]) for a in pred.get(d, []) if a[2] == lbl}
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / len(f1s), dict(zip(labels, f1s))

In [14]:
# ---- class weights: O = 1, rare tags up to 20x ----
counts = torch.zeros(NUM_CLASSES)
for s in train_ds.samples:
    l = s["labels"]; l = l[l != IGNORE_INDEX]
    counts += torch.bincount(l, minlength=NUM_CLASSES).float()
w = (counts.sum() / counts.clamp(min=1)) ** 0.5
w = (w / w[0]).clamp(max=20.0)
print("O fraction:", round((counts[0] / counts.sum()).item(), 4))
criterion = nn.CrossEntropyLoss(weight=w.to(DEVICE), ignore_index=IGNORE_INDEX)

# ---- optimizer with a separate (higher) LR for the head ----
head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
other_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
optimizer = torch.optim.AdamW([
    {"params": other_params, "lr": LEARNING_RATE},
    {"params": head_params,  "lr": HEAD_LR},
], weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup = max(1, int(0.05 * total_steps))
def lr_lambda(s):
    if s < warmup:
        return (s + 1) / warmup
    prog = (s - warmup) / max(1, total_steps - warmup)
    return 0.03 + 0.97 * 0.5 * (1 + math.cos(math.pi * prog))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"Training {EPOCHS} epochs | USE_VAL={USE_VAL}")
best_f1, best_state, best_epoch = -1.0, None, 0

for epoch in range(EPOCHS):
    model.train()
    running, n_log = 0.0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(input_ids)
        loss = criterion(logits.float().view(-1, NUM_CLASSES), labels.view(-1))

        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        running += loss.item(); n_log += 1

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        if (step + 1) % 100 == 0:
            print(f"Epoch [{epoch + 1}/{EPOCHS}] | Step [{step + 1}/{len(train_loader)}] | Loss: {running / n_log:.4f}")
            running, n_log = 0.0, 0

    if USE_VAL and ((epoch + 1) % 2 == 0 or epoch + 1 == EPOCHS):
        vc = {d: doc_logp(t) for d, t in val_texts.items()}
        f1, per = strict_macro_f1(decode_cached(vc, val_texts, 1.5), val_golds)
        print(f"==> Epoch {epoch + 1} | VAL strict macro-F1 (o_bias=1.5): {f1:.4f}")
        print({k: round(v, 3) for k, v in per.items()})
        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch + 1
            best_state = {k: v.detach().clone() for k, v in model.named_parameters() if v.requires_grad}

if USE_VAL and best_state is not None:
    model.load_state_dict(best_state, strict=False)
    print(f"Restored best epoch {best_epoch} (val F1 {best_f1:.4f})")
print("✓ Training finished!")

O fraction: 0.9023
Training 10 epochs | USE_VAL=True
Epoch [1/10] | Step [100/163] | Loss: 1.7122
Epoch [2/10] | Step [100/163] | Loss: 0.3489
==> Epoch 2 | VAL strict macro-F1 (o_bias=1.5): 0.2840
{'APP': 0.073, 'RESP': 0.074, 'A.COUNSEL': 0.097, 'R.COUNSEL': 0.242, 'JUDGE': 0.258, 'WIT': 0.152, 'AUTH': 0.299, 'COURT': 0.689, 'STAT': 0.485, 'PREC': 0.153, 'DATE': 0.727, 'CASENO': 0.159}
Epoch [3/10] | Step [100/163] | Loss: 0.1983
Epoch [4/10] | Step [100/163] | Loss: 0.1274
==> Epoch 4 | VAL strict macro-F1 (o_bias=1.5): 0.3329
{'APP': 0.064, 'RESP': 0.046, 'A.COUNSEL': 0.127, 'R.COUNSEL': 0.279, 'JUDGE': 0.284, 'WIT': 0.143, 'AUTH': 0.301, 'COURT': 0.761, 'STAT': 0.576, 'PREC': 0.216, 'DATE': 0.838, 'CASENO': 0.359}
Epoch [5/10] | Step [100/163] | Loss: 0.0960
Epoch [6/10] | Step [100/163] | Loss: 0.0707
==> Epoch 6 | VAL strict macro-F1 (o_bias=1.5): 0.3525
{'APP': 0.082, 'RESP': 0.052, 'A.COUNSEL': 0.332, 'R.COUNSEL': 0.225, 'JUDGE': 0.33, 'WIT': 0.167, 'AUTH': 0.272, 'COURT': 0.7

In [22]:
val_cache = {d: doc_logp(t) for d, t in val_texts.items()}

# stage 1: O bias x continuation bonus
best = (-1.0, None)
for ob in [0.0, 1.0, 1.5, 2.0, 3.0]:
    for cb in [0.0, 0.5, 1.0, 2.0]:
        f, _ = strict_macro_f1(decode_cached(val_cache, val_texts, ob, cb), val_golds)
        print(f"o_bias={ob} cont_bonus={cb} -> {f:.4f}")
        if f > best[0]:
            best = (f, (ob, cb))
BEST_O_BIAS, BEST_CONT = best[1]
print("best (o_bias, cont_bonus):", best)

# stage 2: merge gap on top of the best setting
BEST_MAX_GAP, best_g = 0, best[0]
for g in [1, 2, 3]:
    f, _ = strict_macro_f1(decode_cached(val_cache, val_texts, BEST_O_BIAS, BEST_CONT, g), val_golds)
    print(f"max_gap={g} -> {f:.4f}")
    if f > best_g:
        best_g, BEST_MAX_GAP = f, g
print(f"\nFINAL SETTINGS: o_bias={BEST_O_BIAS} cont_bonus={BEST_CONT} max_gap={BEST_MAX_GAP} | val F1={best_g:.4f}")

final_val = decode_cached(val_cache, val_texts, BEST_O_BIAS, BEST_CONT, BEST_MAX_GAP)
final_val = {d: [[int(a[0]), int(a[1]), str(a[2])] for a in sp] for d, sp in final_val.items()}
evaluate_predictions_strictly(val_golds, final_val)

o_bias=0.0 cont_bonus=0.0 -> 0.3542
o_bias=0.0 cont_bonus=0.5 -> 0.3623
o_bias=0.0 cont_bonus=1.0 -> 0.3654
o_bias=0.0 cont_bonus=2.0 -> 0.3633
o_bias=1.0 cont_bonus=0.0 -> 0.3576
o_bias=1.0 cont_bonus=0.5 -> 0.3686
o_bias=1.0 cont_bonus=1.0 -> 0.3769
o_bias=1.0 cont_bonus=2.0 -> 0.3821
o_bias=1.5 cont_bonus=0.0 -> 0.3566
o_bias=1.5 cont_bonus=0.5 -> 0.3681
o_bias=1.5 cont_bonus=1.0 -> 0.3787
o_bias=1.5 cont_bonus=2.0 -> 0.3906
o_bias=2.0 cont_bonus=0.0 -> 0.3431
o_bias=2.0 cont_bonus=0.5 -> 0.3664
o_bias=2.0 cont_bonus=1.0 -> 0.3758
o_bias=2.0 cont_bonus=2.0 -> 0.3926
o_bias=3.0 cont_bonus=0.0 -> 0.3231
o_bias=3.0 cont_bonus=0.5 -> 0.3473
o_bias=3.0 cont_bonus=1.0 -> 0.3614
o_bias=3.0 cont_bonus=2.0 -> 0.3890
best (o_bias, cont_bonus): (0.39264665141153593, (2.0, 2.0))
max_gap=1 -> 0.3930
max_gap=2 -> 0.3930
max_gap=3 -> 0.3930

FINAL SETTINGS: o_bias=2.0 cont_bonus=2.0 max_gap=1 | val F1=0.3930


NameError: name 'evaluate_predictions_strictly' is not defined

In [15]:
import time

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def _teacher_pass(ds, batch_size=8):
    model.eval()
    ids, lbls, logits = [], [], []
    for batch in DataLoader(ds, batch_size=batch_size, shuffle=False):
        x = batch["input_ids"].to(DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            out = model(x).float()
        ids.append(batch["input_ids"])
        lbls.append(batch["labels"])
        logits.append(out.cpu().half())
    return torch.cat(ids), torch.cat(lbls), torch.cat(logits)

@torch.no_grad()
def export_teacher_for_kd(path):
    assert USE_VAL, "Run the teacher with USE_VAL=True so fold 2 stays unseen by the teacher"
    fold2_ds = ExactLNERDataset(val_texts, val_golds, tokenizer)
    ids, lbls, lg, fold = [], [], [], []
    for ds, f in [(train_ds, 1), (fold2_ds, 2)]:
        a, b, c = _teacher_pass(ds)
        ids.append(a); lbls.append(b); lg.append(c)
        fold.append(torch.full((len(a),), f, dtype=torch.long))

    sample = train_ds[0]["input_ids"].unsqueeze(0).to(DEVICE)
    model.eval()
    for _ in range(3):
        model(sample)
    _sync()
    t0 = time.time()
    for _ in range(20):
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            model(sample)
    _sync()
    ms_per_window = (time.time() - t0) / 20 * 1000

    torch.save({
        "input_ids": torch.cat(ids),
        "labels": torch.cat(lbls),
        "teacher_logits": torch.cat(lg),
        "fold": torch.cat(fold),
        "itos": tokenizer.itos,
        "num_classes": NUM_CLASSES,
        "ft_mode": globals().get("FT_MODE", "lorma"),
        "teacher_params": sum(p.numel() for p in model.parameters()),
        "teacher_ms_per_window": ms_per_window,
    }, path)
    print("saved", path, "| fold-1 windows:", len(train_ds), "| fold-2 windows:", len(fold2_ds),
          "| teacher ms/window:", round(ms_per_window, 1))

export_teacher_for_kd("/kaggle/working/teacher_kd.pt")

saved /kaggle/working/teacher_kd.pt | fold-1 windows: 652 | fold-2 windows: 660 | teacher ms/window: 26.1


*ARCH-3

In [42]:
import os, json, math, random, time, string, zipfile
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from huggingface_hub import hf_hub_download, login

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)

STUDENT_CKPT = "/kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm"
TOKENIZER_PATH = "/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json"
TEACHER_EXPORT = "/kaggle/working/teacher_kd.pt"

MAX_SEQ_LEN = 512
STRIDE = 128
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
EPOCHS = 10
BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
ALPHA = 0.5
TEMP = 2.0
STUDENT_BIDIRECTIONAL = True
EVAL_O_BIAS = 1.5
EVAL_CONT = 2.0
BEST_O_BIAS = 2.0
BEST_CONT = 2.0
STUDENT_USE_VAL=True

LABEL_NAMES = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
               "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
ID2LABEL = {0: "O"}
LABEL2ID = {"O": 0}
for idx, lbl in enumerate(LABEL_NAMES):
    ID2LABEL[1 + 2 * idx] = f"B-{lbl}"
    LABEL2ID[f"B-{lbl}"] = 1 + 2 * idx
    ID2LABEL[2 + 2 * idx] = f"I-{lbl}"
    LABEL2ID[f"I-{lbl}"] = 2 + 2 * idx
NUM_CLASSES = len(ID2LABEL)
IGNORE_INDEX = -100
print("Device:", DEVICE, "| classes:", NUM_CLASSES)

Device: cuda | classes: 25


In [23]:
def download_phase2_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)

with open(download_phase2_file("Phase_2_LNER/Train/fold2-text.json"), "r", encoding="utf-8") as f:
    val_texts = json.load(f)
with open(download_phase2_file("Phase_2_LNER/Train/fold2-gold.json"), "r", encoding="utf-8") as f:
    val_golds = json.load(f)
with open(download_phase2_file("Phase_2_LNER/Test/fold3-text.json"), "r", encoding="utf-8") as f:
    test_texts = json.load(f)

TEACHER = torch.load(TEACHER_EXPORT, map_location="cpu", weights_only=False)
assert TEACHER["itos"] == tokenizer.itos, "Teacher and student tokenizers differ - distillation would be misaligned"
assert TEACHER["num_classes"] == NUM_CLASSES

fold = TEACHER["fold"]
sel = (fold == 1) if STUDENT_USE_VAL else torch.ones_like(fold, dtype=torch.bool)
HAS_VAL = STUDENT_USE_VAL
print(f"Teacher windows used: {int(sel.sum())} of {len(fold)} | teacher mode: {TEACHER['ft_mode']} | held-out fold 2 available: {HAS_VAL}")

kd_ds = TensorDataset(TEACHER["input_ids"][sel], TEACHER["labels"][sel], TEACHER["teacher_logits"][sel])
train_loader = DataLoader(kd_ds, batch_size=BATCH_SIZE, shuffle=True)

Teacher windows used: 652 of 1312 | teacher mode: lorma | held-out fold 2 available: True


In [38]:
import zipfile, os, glob

source_dir = '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm'
target_pt = '/kaggle/working/decoder_lm3.pt'

pkl = glob.glob(source_dir + '/**/data.pkl', recursive=True)
assert pkl, "No data.pkl under source_dir - this is not an unpacked PyTorch checkpoint"
root = os.path.dirname(pkl[0])
print("checkpoint root:", root)

with zipfile.ZipFile(target_pt, 'w', compression=zipfile.ZIP_STORED) as zf:
    for dirpath, dirs, files in os.walk(root):
        for file in files:
            full_path = os.path.join(dirpath, file)
            rel_path = os.path.relpath(full_path, root)
            zf.write(full_path, arcname="decoder_lm/" + rel_path)

print(f"Created: {target_pt} ({os.path.getsize(target_pt)/1e6:.1f} MB)")
print("valid zip:", zipfile.is_zipfile(target_pt))
print("first entries:", zipfile.ZipFile(target_pt).namelist()[:5])

STUDENT_CKPT = target_pt

checkpoint root: /kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm
Created: /kaggle/working/decoder_lm3.pt (330.1 MB)
valid zip: True
first entries: ['decoder_lm/.format_version', 'decoder_lm/.storage_alignment', 'decoder_lm/data.pkl', 'decoder_lm/version', 'decoder_lm/byteorder']


In [39]:
@dataclass
class GQAConfig:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 3072
    max_seq_len: int = 1024
    dropout: float = 0.1
    norm_first: bool = True

class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.d_model = cfg.d_model
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.num_queries_per_kv = cfg.n_heads // cfg.n_kv_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.n_heads * self.head_dim, cfg.d_model, bias=False)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        if self.num_queries_per_kv > 1:
            k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
            v = v.repeat_interleave(self.num_queries_per_kv, dim=1)
        z = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.attn_dropout.p if self.training else 0.0,
            is_causal=not STUDENT_BIDIRECTIONAL,
        )
        z = z.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(z))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)

class GQABlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = GroupedQueryAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class Arch3NERModel(nn.Module):
    def __init__(self, cfg, num_classes=NUM_CLASSES):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([GQABlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.classifier = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_model),
            nn.LayerNorm(cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, num_classes),
        )

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        return self.classifier(self.ln_f(x))

ckpt = torch.load(STUDENT_CKPT, map_location="cpu", weights_only=False)
state = ckpt["model_state_dict"]
cfg = ckpt["config"]
assert state["tok_emb.weight"].shape[0] == tokenizer.vocab_size, "Checkpoint and tokenizer vocab differ"

model = Arch3NERModel(cfg).to(DEVICE)
backbone = {k: v for k, v in state.items() if not k.startswith("head")}
missing, unexpected = model.load_state_dict(backbone, strict=False)
print("MISSING:", len(missing), missing[:6])
print("UNEXPECTED:", len(unexpected), unexpected[:6])
assert all(m.startswith("classifier.") for m in missing) and not unexpected, "Student backbone did not load"

student_params = sum(p.numel() for p in model.parameters())
print(f"Student params: {student_params:,} | Teacher params: {TEACHER['teacher_params']:,} "
      f"| ratio: {student_params / TEACHER['teacher_params']:.2f}")

MISSING: 6 ['classifier.0.weight', 'classifier.0.bias', 'classifier.1.weight', 'classifier.1.bias', 'classifier.4.weight', 'classifier.4.bias']
UNEXPECTED: 0 []
Student params: 83,123,737 | Teacher params: 112,059,673 | ratio: 0.74


In [40]:
STRIP_CHARS = " \t\n\r" + string.punctuation + "\u2018\u2019\u201c\u201d\u2013\u2014\u2022\u00b7\u2026"

def window_starts(n, max_len=MAX_SEQ_LEN, stride=STRIDE):
    if n <= max_len:
        return [0]
    starts = list(range(0, n - max_len + 1, max_len - stride))
    if starts[-1] + max_len < n:
        starts.append(n - max_len)
    return starts

def clean_span(text, start, end, label):
    while start < end and text[start] in STRIP_CHARS:
        start += 1
    while end > start and text[end - 1] in STRIP_CHARS:
        end -= 1
    return start, end, label

def decode_spans_exact(doc_text, pred_ids, offsets):
    spans, cur_l, cs, ce = [], None, -1, -1

    def flush():
        if cur_l is not None:
            s, e, l = clean_span(doc_text, cs, ce, cur_l)
            if e > s:
                spans.append([s, e, l])

    for t_id, (s_c, e_c) in zip(pred_ids, offsets):
        tag = ID2LABEL.get(int(t_id), "O")
        if tag.startswith("B-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        elif tag.startswith("I-") and cur_l == tag[2:]:
            ce = e_c
        elif tag.startswith("I-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        else:
            flush(); cur_l = None
    flush()
    seen, out = set(), []
    for s in spans:
        if tuple(s) not in seen:
            seen.add(tuple(s)); out.append(s)
    return out

NEG = -1e9
ALLOWED = np.ones((NUM_CLASSES, NUM_CLASSES), dtype=bool)
for cur in range(NUM_CLASSES):
    if ID2LABEL[cur].startswith("I-"):
        for prev in range(NUM_CLASSES):
            ALLOWED[prev, cur] = (ID2LABEL[prev] != "O") and (ID2LABEL[prev][2:] == ID2LABEL[cur][2:])
START_OK = np.array([not ID2LABEL[c].startswith("I-") for c in range(NUM_CLASSES)])

def make_trans(cont_bonus=0.0):
    t = np.where(ALLOWED, 0.0, NEG).astype(np.float64)
    for cur in range(NUM_CLASSES):
        if ID2LABEL[cur].startswith("I-"):
            for prev in range(NUM_CLASSES):
                if ALLOWED[prev, cur]:
                    t[prev, cur] += cont_bonus
    return t

def viterbi(logp, trans):
    T, C = logp.shape
    score = logp[0].copy()
    score[~START_OK] = NEG
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]

@torch.no_grad()
def doc_logp(text):
    model.eval()
    ids, offs = tokenizer.encode_with_exact_offsets(text)
    if not ids:
        return None, None
    acc = torch.zeros(len(ids), NUM_CLASSES)
    cnt = torch.zeros(len(ids), 1)
    for s in window_starts(len(ids)):
        e = min(len(ids), s + MAX_SEQ_LEN)
        chunk = ids[s:e]
        x = torch.tensor([chunk + [0] * (MAX_SEQ_LEN - len(chunk))], dtype=torch.long, device=DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            lp = F.log_softmax(model(x)[0].float(), dim=-1)[: len(chunk)].cpu()
        acc[s:e] += lp
        cnt[s:e] += 1
    return (acc / cnt).numpy(), offs

def decode_cached(cache, texts, o_bias=0.0, cont_bonus=0.0):
    trans, out = make_trans(cont_bonus), {}
    for d, (lp, offs) in cache.items():
        if lp is None:
            out[d] = []
            continue
        lp = lp.astype(np.float64).copy()
        lp[:, 0] += o_bias
        out[d] = decode_spans_exact(texts[d], viterbi(lp, trans), offs)
    return out

def strict_macro_f1(pred, gold, labels=LABEL_NAMES):
    f1s = []
    for lbl in labels:
        tp = fp = fn = 0
        for d in gold:
            g = {(a[0], a[1]) for a in gold[d] if a[2] == lbl}
            p = {(a[0], a[1]) for a in pred.get(d, []) if a[2] == lbl}
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / len(f1s), dict(zip(labels, f1s))

In [43]:
all_labels = TEACHER["labels"][sel]
counts = torch.bincount(all_labels[all_labels != IGNORE_INDEX], minlength=NUM_CLASSES).float()
w = (counts.sum() / counts.clamp(min=1)) ** 0.5
w = (w / w[0]).clamp(max=20.0).to(DEVICE)
print("O fraction:", round((counts[0] / counts.sum()).item(), 4))

def kd_loss(student_logits, teacher_logits, labels):
    C = student_logits.size(-1)
    hard = F.cross_entropy(student_logits.reshape(-1, C), labels.reshape(-1), weight=w, ignore_index=IGNORE_INDEX)
    if ALPHA >= 1.0:
        return hard, hard.detach(), torch.zeros((), device=hard.device)
    mask = labels != IGNORE_INDEX
    tok_w = w[labels.clamp(min=0)] * mask
    s_logp = F.log_softmax(student_logits / TEMP, dim=-1)
    t_prob = F.softmax(teacher_logits / TEMP, dim=-1)
    kl = (t_prob * (torch.log(t_prob.clamp_min(1e-9)) - s_logp)).sum(-1)
    soft = (kl * tok_w).sum() / tok_w.sum().clamp_min(1e-9) * (TEMP ** 2)
    return ALPHA * hard + (1.0 - ALPHA) * soft, hard.detach(), soft.detach()

head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": BACKBONE_LR},
    {"params": head_params, "lr": HEAD_LR},
], weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup = max(1, int(0.05 * total_steps))

def lr_lambda(s):
    if s < warmup:
        return (s + 1) / warmup
    prog = (s - warmup) / max(1, total_steps - warmup)
    return 0.03 + 0.97 * 0.5 * (1 + math.cos(math.pi * prog))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_f1, best_state, best_epoch = -1.0, None, 0
print(f"Distilling | alpha={ALPHA} T={TEMP} | epochs={EPOCHS} | held-out validation: {HAS_VAL}")

for epoch in range(EPOCHS):
    model.train()
    tot, hard_sum, soft_sum, n = 0.0, 0.0, 0.0, 0
    optimizer.zero_grad()
    for step, (x, y, tl) in enumerate(train_loader):
        x, y, tl = x.to(DEVICE), y.to(DEVICE), tl.to(DEVICE).float()
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(x)
        loss, hard, soft = kd_loss(logits.float(), tl, y)
        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        tot += loss.item(); hard_sum += hard.item(); soft_sum += soft.item(); n += 1

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        if (step + 1) % 100 == 0:
            print(f"Epoch [{epoch + 1}/{EPOCHS}] Step [{step + 1}/{len(train_loader)}] "
                  f"loss {tot / n:.4f} | hard {hard_sum / n:.4f} | soft {soft_sum / n:.4f}")
            tot, hard_sum, soft_sum, n = 0.0, 0.0, 0.0, 0

    if HAS_VAL:
        vc = {d: doc_logp(t) for d, t in val_texts.items()}
        f1, per = strict_macro_f1(decode_cached(vc, val_texts, EVAL_O_BIAS, EVAL_CONT), val_golds)
        print(f"==> Epoch {epoch + 1} | VAL strict macro-F1: {f1:.4f}")
        print({k: round(v, 3) for k, v in per.items()})
        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if HAS_VAL and best_state is not None:
    model.load_state_dict(best_state)
    print(f"Restored best epoch {best_epoch} (val F1 {best_f1:.4f})")
print("Distillation finished")

O fraction: 0.9023
Distilling | alpha=0.5 T=2.0 | epochs=10 | held-out validation: True
Epoch [1/10] Step [100/163] loss 2.4632 | hard 1.3760 | soft 3.5503
==> Epoch 1 | VAL strict macro-F1: 0.1110
{'APP': 0.0, 'RESP': 0.009, 'A.COUNSEL': 0.0, 'R.COUNSEL': 0.0, 'JUDGE': 0.018, 'WIT': 0.0, 'AUTH': 0.164, 'COURT': 0.582, 'STAT': 0.18, 'PREC': 0.027, 'DATE': 0.281, 'CASENO': 0.071}
Epoch [2/10] Step [100/163] loss 2.2213 | hard 1.2603 | soft 3.1823
==> Epoch 2 | VAL strict macro-F1: 0.1059
{'APP': 0.005, 'RESP': 0.007, 'A.COUNSEL': 0.0, 'R.COUNSEL': 0.0, 'JUDGE': 0.027, 'WIT': 0.058, 'AUTH': 0.158, 'COURT': 0.619, 'STAT': 0.168, 'PREC': 0.006, 'DATE': 0.197, 'CASENO': 0.025}
Epoch [3/10] Step [100/163] loss 2.0192 | hard 1.1571 | soft 2.8814
==> Epoch 3 | VAL strict macro-F1: 0.1334
{'APP': 0.0, 'RESP': 0.014, 'A.COUNSEL': 0.0, 'R.COUNSEL': 0.0, 'JUDGE': 0.035, 'WIT': 0.07, 'AUTH': 0.2, 'COURT': 0.587, 'STAT': 0.212, 'PREC': 0.03, 'DATE': 0.389, 'CASENO': 0.063}
Epoch [4/10] Step [100/163

In [44]:
if HAS_VAL:
    val_cache = {d: doc_logp(t) for d, t in val_texts.items()}
    best = (-1.0, None)
    for ob in [1.0, 1.5, 2.0, 2.5, 3.0]:
        for cb in [0.0, 1.0, 2.0, 3.0, 4.0]:
            f, _ = strict_macro_f1(decode_cached(val_cache, val_texts, ob, cb), val_golds)
            print(f"o_bias={ob} cont_bonus={cb} -> {f:.4f}")
            if f > best[0]:
                best = (f, (ob, cb))
    BEST_O_BIAS, BEST_CONT = best[1]
    f, per = strict_macro_f1(decode_cached(val_cache, val_texts, BEST_O_BIAS, BEST_CONT), val_golds)
    print(f"\nStudent tuned val strict macro-F1: {f:.4f} at o_bias={BEST_O_BIAS}, cont_bonus={BEST_CONT}")
    print({k: round(v, 3) for k, v in per.items()})

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

model.eval()
sample = TEACHER["input_ids"][0:1].to(DEVICE)
with torch.no_grad():
    for _ in range(3):
        model(sample)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    _sync()
    t0 = time.time()
    for _ in range(20):
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            model(sample)
    _sync()
student_ms = (time.time() - t0) / 20 * 1000
peak_mb = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else float("nan")

print("\nEfficiency")
print(f"  parameters:  student {student_params / 1e6:.1f}M | teacher {TEACHER['teacher_params'] / 1e6:.1f}M")
print(f"  ms/window:   student {student_ms:.1f} | teacher {TEACHER['teacher_ms_per_window']:.1f}")
print(f"  student peak inference memory: {peak_mb:.0f} MB")

o_bias=1.0 cont_bonus=0.0 -> 0.1129
o_bias=1.0 cont_bonus=1.0 -> 0.1325
o_bias=1.0 cont_bonus=2.0 -> 0.1462
o_bias=1.0 cont_bonus=3.0 -> 0.1533
o_bias=1.0 cont_bonus=4.0 -> 0.1579
o_bias=1.5 cont_bonus=0.0 -> 0.1088
o_bias=1.5 cont_bonus=1.0 -> 0.1303
o_bias=1.5 cont_bonus=2.0 -> 0.1496
o_bias=1.5 cont_bonus=3.0 -> 0.1571
o_bias=1.5 cont_bonus=4.0 -> 0.1636
o_bias=2.0 cont_bonus=0.0 -> 0.1066
o_bias=2.0 cont_bonus=1.0 -> 0.1284
o_bias=2.0 cont_bonus=2.0 -> 0.1491
o_bias=2.0 cont_bonus=3.0 -> 0.1617
o_bias=2.0 cont_bonus=4.0 -> 0.1694
o_bias=2.5 cont_bonus=0.0 -> 0.1030
o_bias=2.5 cont_bonus=1.0 -> 0.1234
o_bias=2.5 cont_bonus=2.0 -> 0.1466
o_bias=2.5 cont_bonus=3.0 -> 0.1660
o_bias=2.5 cont_bonus=4.0 -> 0.1753
o_bias=3.0 cont_bonus=0.0 -> 0.0982
o_bias=3.0 cont_bonus=1.0 -> 0.1180
o_bias=3.0 cont_bonus=2.0 -> 0.1431
o_bias=3.0 cont_bonus=3.0 -> 0.1654
o_bias=3.0 cont_bonus=4.0 -> 0.1771

Student tuned val strict macro-F1: 0.1771 at o_bias=3.0, cont_bonus=4.0
{'APP': 0.0, 'RESP': 0.029,

In [45]:
test_cache = {d: doc_logp(t) for d, t in test_texts.items()}
preds = decode_cached(test_cache, test_texts, BEST_O_BIAS, BEST_CONT)
preds = {d: [[int(a[0]), int(a[1]), str(a[2])] for a in sp] for d, sp in preds.items()}

output_json = "/kaggle/working/prediction.json"
output_zip = "/kaggle/working/submission.zip"
with open(output_json, "w", encoding="utf-8") as f:
    f.write(json.dumps(preds))
    f.flush()
    os.fsync(f.fileno())
if os.path.exists(output_zip):
    os.remove(output_zip)
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(output_json, arcname="prediction.json")

torch.save(model.state_dict(), "/kaggle/working/student_arch3_kd.pt")
print("submission.zip ready:", os.path.getsize(output_zip), "bytes | student weights saved")

submission.zip ready: 39429 bytes | student weights saved


In [23]:
# BEST_O_BIAS, BEST_CONT, BEST_MAX_GAP come from Cell 7 (or set them by hand, e.g. 2.0, 0.5, 0)
test_cache = {d: doc_logp(t) for d, t in test_texts.items()}
test_predictions = decode_cached(test_cache, test_texts, BEST_O_BIAS, BEST_CONT, BEST_MAX_GAP)
test_predictions = {d: [[int(a[0]), int(a[1]), str(a[2])] for a in sp] for d, sp in test_predictions.items()}

output_json = "/kaggle/working/prediction.json"
output_zip = "/kaggle/working/submission.zip"
with open(output_json, "w", encoding="utf-8") as f:
    f.write(json.dumps(test_predictions))
    f.flush()
    os.fsync(f.fileno())
if os.path.exists(output_zip):
    os.remove(output_zip)
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(output_json, arcname="prediction.json")
print(f"\n✓ Submission zip created: {output_zip} ({os.path.getsize(output_zip):,} bytes)")

with zipfile.ZipFile(output_zip, "r") as zf:
    verified = json.loads(zf.read("prediction.json").decode("utf-8"))
    first_k = list(verified.keys())[0]
    print(f"✓ ZIP VERIFIED: Found {len(verified)} documents.")
    print(f"Sample prediction for doc {first_k}: {verified[first_k][:3]}")


✓ Submission zip created: /kaggle/working/submission.zip (29,614 bytes)
✓ ZIP VERIFIED: Found 35 documents.
Sample prediction for doc 1128835: [[18, 40, 'COURT'], [70, 101, 'CASENO'], [114, 162, 'CASENO']]
